# Notebook 07: Hyperparameter Optimization (HPO)

**MoEForecaster**: a Mixture-of-Experts (MoE) model for hourly electricity demand forecasting in Colombia's National Interconnected System (SIN).

---

## Objective

This notebook tunes the two groups of hyperparameters of the MoE system:

1. **Expert hyperparameters.** For every expert (one per cluster, i.e. per (day type, week type) combination), the model family selected in Notebook 05 is tuned with Random Search combined with Successive Halving (`HalvingRandomSearchCV`) under time-series cross-validation. The notebook produces convergence curves per expert, a table of the best hyperparameters and parallel-coordinate plots of the best candidates.
2. **Gating hyperparameters.** The confidence threshold $\theta$ and the softmax temperature $\tau$ of a distance-based adaptive gating are selected by an exhaustive grid search on a temporal hold-out set.

## Inputs

| File | Produced by | Content |
|---|---|---|
| `data/processed/features.parquet` | Notebook 02 | Hourly feature matrix (calendar, holiday and lag features) and the target `demand` |
| `data/clustering/cluster_assignments.parquet` | Notebook 03 | Hourly cluster assignments (`datetime`, `cluster_day`, `cluster_week`, `expert_id`) |
| `results/05_expert_selection_best.csv` | Notebook 05 | Best model family per expert (one row per `expert_id`, sorted by `expert_id`) |

## Outputs

| File | Content | Used by |
|---|---|---|
| `models/hyperparameters/best_hyperparameters.json` | Best hyperparameters, model family and best cross-validated RMSE per expert | Notebook 08 |
| `models/hyperparameters/best_gating_params.json` | Selected gating threshold $\theta$, temperature $\tau$, `min_prob` and hold-out metrics | Notebook 08 |
| `models/hyperparameters/expert_<id>_optimised.pkl` | Best estimator of each expert, refitted on all samples of its cluster | Reference artifact |
| `results/07_hyperparameter_search_spaces.csv` | Search spaces of the `bayesian_ridge`, `lasso` and `lgbm` families | Documentation |
| `results/07_hpo_results.csv` | HPO summary per expert (RMSE, MAPE, number of candidates, run time, best parameters) | Documentation |
| `results/07_gating_grid_results.csv` | Hold-out metrics of every $(\theta, \tau)$ combination | Documentation |
| `results/07_hpo_convergence_curves.png` | Convergence curves of the successive-halving search per expert | Figure |
| `results/07_hpo_parallel_<family>.png` | Parallel-coordinate plots of the top-10 candidates per model family | Figure |
| `results/07_hpo_gating_heatmap.png` | MASE and MAPE heat maps over the gating grid | Figure |

## Place in the pipeline

01 data acquisition → 02 preprocessing and feature engineering → 03 clustering → 04 cluster analysis and classifiers → 05 expert selection → 06 gating design → **07 hyperparameter optimization** → 08 production assembly → 09 benchmark.

Notebook 08 reads the best expert hyperparameters and the gating parameters written here to assemble the production `MoEForecaster`.

## Environment setup

This notebook runs unchanged on a local machine (pip/venv or conda), in Docker and in Google Colab.

**`OUTPUT_ROOT`** (next cell) is the folder where the pipeline stores `data/`, `models/` and `results/`:

| Value | Where artifacts are stored |
|---|---|
| `None` (default) | The repository folder, or the folder in the `MOE_OUTPUT_ROOT` environment variable if it is set |
| `"/content/drive/MyDrive/MoEForecaster"` | Google Drive (Colab only; Drive is mounted automatically) |
| `"/any/absolute/path"` or `"relative/path"` | Any folder; relative paths are resolved from the repository root |

Use the **same `OUTPUT_ROOT` in every notebook**: each notebook reads the artifacts written by the previous ones. Setting the `MOE_OUTPUT_ROOT` environment variable once is the easiest way to keep it consistent.

The cell after it finds the repository, which it clones first when running in Colab. In Colab it also installs the pinned dependencies from `requirements.txt` and makes the `moe_forecaster` package in `src/` importable.

In [ ]:
# ============================================================
# PARAMETERS: the only cell you may need to edit
# ============================================================
# Folder for data/, models/ and results/ (see the table above).
# None -> repository folder, or $MOE_OUTPUT_ROOT if that variable is set.
OUTPUT_ROOT = None

In [ ]:
# ============================================================
# ENVIRONMENT BOOTSTRAP: no edits needed
# ============================================================
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/astorress/MoEForecaster.git"
IN_COLAB = "google.colab" in sys.modules


def find_repo_root(start):
    """Walk up from `start` until the folder that contains src/moe_forecaster/."""
    for folder in [start, *start.parents]:
        if (folder / "src" / "moe_forecaster").is_dir():
            return folder
    return None


REPO_ROOT = find_repo_root(Path.cwd().resolve())

if REPO_ROOT is None and IN_COLAB:
    # Colab opens the notebook on its own, without the rest of the repository
    REPO_ROOT = Path("/content/MoEForecaster")
    if not REPO_ROOT.exists():
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_ROOT)], check=True)

if REPO_ROOT is None:
    raise RuntimeError(
        "Repository root not found. Start Jupyter from inside the MoEForecaster folder "
        "(local, conda or Docker), or open the notebook in Google Colab."
    )

if IN_COLAB:
    # Install the pinned versions once per Colab runtime. Colab's preinstalled
    # PyTorch (built with CUDA) is kept, so the torch pin is skipped here.
    marker = Path("/content/.moe_requirements_installed")
    if not marker.exists():
        reqs = [
            line for line in (REPO_ROOT / "requirements.txt").read_text().splitlines()
            if line.strip() and not line.startswith(("#", "torch"))
        ]
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *reqs], check=True)
        marker.touch()
        print("Dependencies installed. If Colab asks to restart the session, restart it and run the notebook again.")

if str(REPO_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / "src"))

from moe_forecaster.config import setup_paths

DATA_DIR, MODELS_DIR, RESULTS_DIR = setup_paths(OUTPUT_ROOT, repo_root=REPO_ROOT)

## 1. Configuration

Imports, random seed, number of parallel jobs and the plotting style used for all figures of this notebook.

In [ ]:
import os, json, warnings, time, pickle
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

from scipy.stats import uniform, loguniform, randint

from sklearn.experimental import enable_halving_search_cv  # noqa: F401
from sklearn.model_selection import (
    HalvingRandomSearchCV,
    TimeSeriesSplit,
)
from sklearn.linear_model import Ridge, ElasticNet, Lasso, BayesianRidge, SGDRegressor
from sklearn.ensemble import (
    RandomForestRegressor,
    ExtraTreesRegressor,
    GradientBoostingRegressor,
)
from sklearn.metrics import mean_absolute_error, make_scorer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

import lightgbm as lgb
import xgboost as xgb

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

# ============================================================
# CONFIGURATION
# ============================================================
RANDOM_SEED = 42
N_JOBS      = -1

os.makedirs(f"{RESULTS_DIR}/figures", exist_ok=True)

# ── Publication-quality plotting style ─────────────────────────────────────
plt.rcParams.update({
    "figure.dpi": 150,
    "savefig.dpi": 300,
    "font.size": 11,
    "axes.titlesize": 13,
    "axes.labelsize": 11,
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
    "legend.fontsize": 9,
    "figure.figsize": (10, 5),
    "axes.grid": True,
    "grid.alpha": 0.3,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

PALETTE = sns.color_palette("colorblind")

np.random.seed(RANDOM_SEED)
print(f"Configuration loaded: seed={RANDOM_SEED}, n_jobs={N_JOBS}")

---

## 2. Load artifacts from previous notebooks

Three artifacts are loaded:

| Artifact | Source | Description |
|-----------|--------|-------------|
| `features_df` | Notebook 02 (`data/processed/features.parquet`) | Hourly feature matrix with all engineered features and the target `demand` |
| `cluster_df` | Notebook 03 (`data/clustering/cluster_assignments.parquet`) | Hourly mapping `datetime → expert_id`, converted here to an integer `cluster_id` |
| `best_models` | Notebook 05 (`results/05_expert_selection_best.csv`) | Dictionary `cluster_id → model family` (best model per expert) |

If any artifact is missing, the notebook stops with an error: no synthetic fallback data are used.

### 2.1 Feature matrix

In [ ]:
# ── 2.1  Feature matrix ──────────────────────────────────────────────────
features_path = Path(DATA_DIR) / "processed" / "features.parquet"
if features_path.exists():
    features_df = pd.read_parquet(features_path)
    print(f"Features loaded: {features_df.shape}")
else:
    # No synthetic fallback data: if an artifact from a previous notebook is
    # missing, that notebook must be run first.
    raise FileNotFoundError(f"{features_path} not found. Run Notebook 02 first (make pipeline).")

### 2.2 Exclusion of 2020 and 2021

The years 2020 and 2021 are removed from the feature matrix, as in Notebooks 04, 05, 06 and 09, so that hyperparameters are tuned on the same period used for expert selection. This period coincides with the COVID-19 pandemic.

In [ ]:
features_df = features_df[~features_df.index.year.isin([2020, 2021])]

### 2.3 Cluster assignments

Each hourly assignment carries its own timestamp (`datetime` column written by Notebook 03), which is used as the index for the join with the feature matrix. The textual `expert_id` labels (`"<day cluster>_<week cluster>"`) are mapped to consecutive integers `cluster_id` in lexicographic order of `expert_id`.

In [ ]:
# ── 2.3  Cluster assignments ─────────────────────────────────────────────
cluster_path = Path(DATA_DIR) / "clustering" / "cluster_assignments.parquet"
if cluster_path.exists():
    cluster_df = pd.read_parquet(cluster_path)
    print(f"Cluster assignments loaded: {cluster_df.shape}")
    # Use the 'datetime' column as the index, consistent with features_df
    if "datetime" in cluster_df.columns:
        cluster_df.set_index("datetime", inplace=True)
        cluster_df.index.name = "datetime"  # standardize the index name

    if "expert_id" in cluster_df.columns:
        # Map the textual expert_ids to integer cluster_ids for internal processing and sorting
        unique_expert_ids = sorted(cluster_df["expert_id"].unique())
        expert_id_to_cluster_id = {eid: i for i, eid in enumerate(unique_expert_ids)}
        cluster_df["cluster_id"] = cluster_df["expert_id"].map(expert_id_to_cluster_id)
        # Keep the inverse mapping for debugging/visualization
        cluster_id_to_expert_id = {i: eid for i, eid in enumerate(unique_expert_ids)}
        print(f"Textual expert_ids mapped to integer cluster_ids: {cluster_id_to_expert_id}")
    elif "cluster_id" in cluster_df.columns:
        print("Using the existing 'cluster_id' column of the parquet file.")
    else:
        raise ValueError("No valid cluster identifier column ('expert_id' or 'cluster_id') found in the loaded cluster_df.")

else:
    # No synthetic fallback data: if an artifact from a previous notebook is
    # missing, that notebook must be run first.
    raise FileNotFoundError(f"{cluster_path} not found. Run Notebook 03 first (make pipeline).")


# Each assignment carries its own timestamp (datetime column, see Notebook 03). Rebuilding it
# from the row position shifted it by 48 h, because the clustering drops the first incomplete ISO week.
cluster_df['timestamp'] = cluster_df.index
cluster_df = cluster_df.reset_index(drop=True)

print(f"\nCluster table shape: {cluster_df.shape}")
print(f"Cluster table columns: {list(cluster_df.columns)}")

# Make sure the timestamp is a datetime
cluster_df['timestamp'] = pd.to_datetime(cluster_df['timestamp'])

cluster_df.index = cluster_df['timestamp']
cluster_df.drop(columns=['timestamp'], inplace=True)

In [ ]:
cluster_df.head()

### 2.4 Best model family per expert

`results/05_expert_selection_best.csv` has one row per expert, sorted by `expert_id`. The row position (0, 1, 2, ...) is used as the expert key, so that it matches the integer `cluster_id` defined above (both follow the lexicographic order of `expert_id`).

In [ ]:
# ── 2.4  Best model family per expert ────────────────────────────────────
best_models_path = Path(RESULTS_DIR) / "05_expert_selection_best.csv"

if best_models_path.exists():
    df_aux = pd.read_csv(best_models_path)
    # Key = row position, which matches the integer cluster_id (both sorted by expert_id)
    best_models = dict(zip(df_aux.index, df_aux.model))

    print(f"Best models loaded: {len(best_models)} experts")
else:
    # No synthetic fallback data: if an artifact from a previous notebook is
    # missing, that notebook must be run first.
    raise FileNotFoundError(f"{best_models_path} not found. Run Notebook 05 first (make pipeline).")

### 2.5 Feature set and target

The experts use the same feature set as in Notebook 05: calendar and holiday features plus the autoregressive demand lags (`demand_lag_*`). The target is the hourly demand `demand`. The feature matrix is then joined with `cluster_id` on the timestamp index (inner join).

In [ ]:
# ── 2.5  Feature set: calendar/holiday features + demand lags ─────────────
calendar_only = [
    'hour_sin', 'hour_cos', 'dayofweek_sin', 'dayofweek_cos', 'month_sin', 'month_cos',
    'is_holiday', 'is_bridge', 'days_to_next_holiday', 'days_from_prev_holiday',
]

all_cols = list(features_df.columns)

# Lag features
lag_features = sorted([c for c in all_cols if c.startswith('demand_lag_')])

calendar_lags = calendar_only + lag_features

In [ ]:
# ── 2.5  Join features + cluster_id ──────────────────────────────────────
# Both features_df and cluster_df are indexed by the hourly timestamp.
if "cluster_id" not in features_df.columns:
    features_df = features_df.join(cluster_df[["cluster_id"]], how="inner")

TARGET = "demand"
#FEATURE_COLS = [c for c in features_df.columns if c not in [TARGET, "cluster_id"]]
FEATURE_COLS = calendar_lags

print(f"\nFinal dataset: {features_df.shape[0]:,} rows, {len(FEATURE_COLS)} features")
print(f"Experts (clusters): {sorted(features_df['cluster_id'].unique())}")
print(f"Model assignments: {best_models}")

---

## 3. Hyperparameter search spaces per model family

Each model family has its own search space. Log-uniform distributions are used for scale parameters (learning rate, regularization strength), and uniform or discrete-uniform distributions for structural parameters (depth, number of leaves, number of estimators, sampling fractions).

Linear models are wrapped in a `Pipeline` with mean imputation and standardization, so that the regularization penalty acts on comparable scales; tree-based models receive the raw features. Every model uses the fixed seed `RANDOM_SEED` and a single thread (`n_jobs=1`).

The second cell exports the search spaces of the `bayesian_ridge`, `lasso` and `lgbm` families to `results/07_hyperparameter_search_spaces.csv` for documentation.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# Search space definitions
# ══════════════════════════════════════════════════════════════════════════

SEARCH_SPACES: dict[str, dict] = {
    # ── Linear family ────────────────────────────────────────────────────
    "ridge": {
        "model": Pipeline([
            ("imputer", SimpleImputer(strategy="mean")),
            ("scaler", StandardScaler()),
            ("reg", Ridge(random_state=RANDOM_SEED)),
        ]),
        "params": {
            "reg__alpha": loguniform(1e-3, 1e3),
        },
    },
    "lasso": {
        "model": Pipeline([
            ("imputer", SimpleImputer(strategy="mean")),
            ("scaler", StandardScaler()),
            ("reg", Lasso(random_state=RANDOM_SEED, max_iter=5000)),
        ]),
        "params": {
            "reg__alpha": loguniform(1e-4, 1e2),
        },
    },
    "elasticnet": {
        "model": Pipeline([
            ("imputer", SimpleImputer(strategy="mean")),
            ("scaler", StandardScaler()),
            ("reg", ElasticNet(random_state=RANDOM_SEED, max_iter=5000)),
        ]),
        "params": {
            "reg__alpha":    loguniform(1e-4, 1e2),
            "reg__l1_ratio": uniform(0.01, 0.98),  # [0.01, 0.99]
        },
    },
    "bayesian_ridge": {
        "model": Pipeline([
            ("imputer", SimpleImputer(strategy="mean")),
            ("scaler", StandardScaler()),
            ("reg", BayesianRidge()),
        ]),
        "params": {
            "reg__alpha_1":  loguniform(1e-7, 1e-1),
            "reg__alpha_2":  loguniform(1e-7, 1e-1),
            "reg__lambda_1": loguniform(1e-7, 1e-1),
            "reg__lambda_2": loguniform(1e-7, 1e-1),
        },
    },
    "sgd": {
        "model": Pipeline([
            ("imputer", SimpleImputer(strategy="mean")),
            ("scaler", StandardScaler()),
            ("reg", SGDRegressor(random_state=RANDOM_SEED, max_iter=5000)),
        ]),
        "params": {
            "reg__alpha":    loguniform(1e-6, 1e-1),
            "reg__l1_ratio": uniform(0.0, 1.0),
            "reg__penalty":  ["l2", "l1", "elasticnet"],
        },
    },

    # ── Boosting family ──────────────────────────────────────────────────
    "lgbm": {
        "model": lgb.LGBMRegressor(
            random_state=RANDOM_SEED, n_jobs=1, verbosity=-1,
        ),
        "params": {
            "learning_rate":     loguniform(1e-3, 3e-1),
            "n_estimators":      randint(100, 1500),
            "max_depth":         randint(3, 12),
            "num_leaves":        randint(15, 255),
            "min_child_samples": randint(5, 100),
            "subsample":         uniform(0.5, 0.5),        # [0.5, 1.0]
            "colsample_bytree":  uniform(0.4, 0.6),        # [0.4, 1.0]
            "reg_alpha":         loguniform(1e-4, 1e1),
            "reg_lambda":        loguniform(1e-4, 1e1),
        },
    },
    "xgboost": {
        "model": xgb.XGBRegressor(
            random_state=RANDOM_SEED, n_jobs=1, verbosity=0,
            tree_method="hist",
        ),
        "params": {
            "learning_rate":     loguniform(1e-3, 3e-1),
            "n_estimators":      randint(100, 1500),
            "max_depth":         randint(3, 12),
            "min_child_weight":  randint(1, 30),
            "subsample":         uniform(0.5, 0.5),
            "colsample_bytree":  uniform(0.4, 0.6),
            "reg_alpha":         loguniform(1e-4, 1e1),
            "reg_lambda":        loguniform(1e-4, 1e1),
            "gamma":             loguniform(1e-4, 1e1),
        },
    },
    "gradient_boosting": {
        "model": GradientBoostingRegressor(random_state=RANDOM_SEED),
        "params": {
            "learning_rate":    loguniform(1e-3, 3e-1),
            "n_estimators":     randint(100, 1000),
            "max_depth":        randint(3, 10),
            "min_samples_split":randint(2, 50),
            "min_samples_leaf": randint(1, 30),
            "subsample":        uniform(0.5, 0.5),
            "max_features":     uniform(0.3, 0.7),
        },
    },

    # ── Forest family ────────────────────────────────────────────────────
    "random_forest": {
        "model": RandomForestRegressor(random_state=RANDOM_SEED, n_jobs=1),
        "params": {
            "n_estimators":     randint(100, 1000),
            "max_depth":        randint(5, 30),
            "min_samples_split":randint(2, 30),
            "min_samples_leaf": randint(1, 20),
            "max_features":     uniform(0.3, 0.7),
        },
    },
    "extra_trees": {
        "model": ExtraTreesRegressor(random_state=RANDOM_SEED, n_jobs=1),
        "params": {
            "n_estimators":     randint(100, 1000),
            "max_depth":        randint(5, 30),
            "min_samples_split":randint(2, 30),
            "min_samples_leaf": randint(1, 20),
            "max_features":     uniform(0.3, 0.7),
        },
    },
}

print("Search spaces defined for:")
for name, spec in SEARCH_SPACES.items():
    n_hp = len(spec["params"])
    print(f"  {name:22s}  {n_hp} hyperparameters")

In [ ]:
# Export the search spaces of selected families to CSV (documentation only)
param_specs = []
models_to_export = ["bayesian_ridge", "lasso", "lgbm"]

for model_name in models_to_export:
    if model_name in SEARCH_SPACES:
        params = SEARCH_SPACES[model_name]["params"]
        for param_name, distribution in params.items():
            entry = {"model": model_name, "parameter": param_name}

            # Check if it's a scipy.stats frozen distribution
            if hasattr(distribution, 'dist') and hasattr(distribution.dist, 'name'):
                dist_name = distribution.dist.name
                if dist_name == 'uniform':
                    entry["type"] = "uniform"
                    # For uniform(loc, scale), loc is args[0], scale is args[1]
                    entry["min_value"] = distribution.args[0]
                    entry["max_value"] = distribution.args[0] + distribution.args[1]
                elif dist_name == 'loguniform':
                    entry["type"] = "loguniform"
                    # loguniform(a, b) where a and b are min and max
                    entry["min_value"] = distribution.a
                    entry["max_value"] = distribution.b
                elif dist_name == 'randint':
                    entry["type"] = "randint"
                    # randint(low, high), high is exclusive
                    entry["min_value"] = distribution.a
                    entry["max_value"] = distribution.b - 1
                else:
                    entry["type"] = str(type(distribution))
                    entry["min_value"] = "N/A"
                    entry["max_value"] = "N/A"
            elif isinstance(distribution, list):
                entry["type"] = "categorical"
                entry["min_value"] = "N/A"
                entry["max_value"] = "N/A"
                entry["categories"] = str(distribution)
            else:
                entry["type"] = str(type(distribution))
                entry["min_value"] = "N/A"
                entry["max_value"] = "N/A"
            param_specs.append(entry)

df_search_spaces = pd.DataFrame(param_specs)

# Save to CSV
csv_path = Path(RESULTS_DIR) / "07_hyperparameter_search_spaces.csv"
df_search_spaces.to_csv(csv_path, index=False)
print(f"Hyperparameter search spaces saved to: {csv_path}")
display(df_search_spaces)

---

## 4. Per-expert HPO with `HalvingRandomSearchCV`

For each expert (cluster) the following steps are performed:

1. Select the hourly samples assigned to the cluster, in chronological order.
2. Look up the model family selected for that expert in Notebook 05.
3. Run `HalvingRandomSearchCV` on the corresponding search space. Successive halving draws `N_CANDIDATES` random configurations, evaluates them on a small number of samples (`min_resources`) and, at each round, keeps the best $1/\text{factor}$ of the candidates while multiplying the number of samples by `factor`. This allocates most of the computational budget to the most promising configurations.
4. Cross-validation uses `TimeSeriesSplit` with 3 folds, so that every validation fold lies after its training fold in time. The selection metric is RMSE (`neg_root_mean_squared_error`). After the search, the best estimator is refitted on all samples of the cluster (`refit=True`), and its MAPE is computed with the same cross-validation scheme.
5. Record the best parameters, the best RMSE and MAPE and the full cross-validation results.

If a cluster has fewer than `2 × MIN_RESOURCES` samples, `min_resources` is reduced to `max(50, n // 10)`.

In [ ]:
# ── Cross-validation strategy ────────────────────────────────────────────
tscv = TimeSeriesSplit(n_splits=3, gap=0)  # gap=0: no gap between training and validation folds

# ── HPO configuration ────────────────────────────────────────────────────
N_CANDIDATES   = 200
HALVING_FACTOR = 3
MIN_RESOURCES  = 100
SCORING        = "neg_root_mean_squared_error"

print(f"TimeSeriesSplit: {tscv.n_splits} folds, gap={tscv.gap}")
print(f"HalvingRandomSearch: n_candidates={N_CANDIDATES}, factor={HALVING_FACTOR}")
print(f"  min_resources={MIN_RESOURCES}, scoring={SCORING}")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# Run HPO for every expert
# ══════════════════════════════════════════════════════════════════════════
from sklearn.base import clone
from sklearn.model_selection import cross_val_score

hpo_results = {}       # expert_id -> {best_params, best_score, cv_results, search_obj}
hpo_summary_rows = []  # rows of the final summary table

for expert_id, model_name in sorted(best_models.items(), key=lambda x: int(x[0])):
    cid = int(expert_id)
    print(f"\n{'='*70}")
    print(f"Expert {expert_id}: model {model_name}")
    print(f"{'='*70}")

    # ── Select the samples of the cluster ───────────────────────────────────
    mask = features_df["cluster_id"] == cid
    df_cluster = features_df.loc[mask].sort_index()
    print(f"  Cluster {cid}: {len(df_cluster):,} samples")

    if len(df_cluster) < MIN_RESOURCES * 2:
        print(f"  WARNING: too few samples ({len(df_cluster)}). Reducing min_resources.")
        effective_min = max(50, len(df_cluster) // 10)
    else:
        effective_min = MIN_RESOURCES

    X = df_cluster[FEATURE_COLS].values
    y = df_cluster[TARGET].values

    # ── Retrieve the search space ─────────────────────────────────────────
    if model_name not in SEARCH_SPACES:
        print(f"  SKIPPED: no search space defined for '{model_name}'.")
        continue

    spec  = SEARCH_SPACES[model_name]
    model = clone(spec["model"])
    param_distributions = spec["params"]

    # ── HalvingRandomSearchCV ─────────────────────────────────────────
    search = HalvingRandomSearchCV(
        estimator=model,
        param_distributions=param_distributions,
        n_candidates=N_CANDIDATES,
        factor=HALVING_FACTOR,
        resource="n_samples",
        min_resources=effective_min,
        cv=tscv,
        scoring=SCORING,
        random_state=RANDOM_SEED,
        n_jobs=1,  # changed from N_JOBS to 1 while debugging a NaN-score issue
        verbose=0,
        refit=True,
    )

    t0 = time.time()
    search.fit(X, y)
    elapsed = time.time() - t0

    best_idx = search.best_index_
    best_rmse = -search.cv_results_["mean_test_score"][best_idx]

    # Compute the MAPE of the best model with cross_val_score (same CV folds) to report both metrics
    best_model = search.best_estimator_
    mape_scores = cross_val_score(best_model, X, y, cv=tscv, scoring="neg_mean_absolute_percentage_error")
    best_mape = -mape_scores.mean()

    print(f"  Best RMSE : {best_rmse:,.2f} kWh")
    print(f"  Best MAPE : {best_mape:.4%}")
    print(f"  Time      : {elapsed:.1f} s")
    print(f"  Best parameters:")
    for k, v in search.best_params_.items():
        if isinstance(v, float):
            print(f"    {k}: {v:.6f}")
        else:
            print(f"    {k}: {v}")

    # ── Store results ─────────────────────────────────────────────────
    hpo_results[expert_id] = {
        "model_name":  model_name,
        "best_params":  search.best_params_,
        "best_score":   best_rmse,
        "cv_results":   pd.DataFrame(search.cv_results_),
        "elapsed_s":    elapsed,
        "search_obj":   search,
    }

    hpo_summary_rows.append({
        "expert_id":  expert_id,
        "model":      model_name,
        "best_rmse":   best_rmse,
        "best_mape":   best_mape,
        "n_candidates_evaluated": len(search.cv_results_["params"]),
        "elapsed_s":  round(elapsed, 1),
        **{k: v for k, v in search.best_params_.items()},
    })

print(f"\nHPO completed for {len(hpo_results)} experts.")

### 4.1 Convergence curves: RMSE per halving round

Each panel shows, for one expert, how the **mean cross-validated RMSE** of the candidates evolves across the successive halving rounds (error bars: standard deviation across candidates; `n` = number of candidates evaluated in the round). The early rounds evaluate many candidates on few samples; the later rounds evaluate fewer candidates on progressively more samples. The dashed line marks the RMSE of the selected configuration.

In [ ]:
n_experts = len(hpo_results)
if n_experts == 0:
    print("No HPO results to plot.")
else:
    ncols = min(3, n_experts)
    nrows = int(np.ceil(n_experts / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(5.5 * ncols, 4.5 * nrows),
                              squeeze=False)

    for idx, (expert_id, res) in enumerate(sorted(hpo_results.items(),
                                                    key=lambda x: int(x[0]))):
        ax = axes[idx // ncols, idx % ncols]
        cv_df = res["cv_results"].copy()

        # HalvingRandomSearchCV stores the column 'iter' with the halving round
        if "iter" in cv_df.columns:
            round_col = "iter"
        elif "n_resources" in cv_df.columns:
            round_col = "n_resources"
        else:
            round_col = None

        cv_df["rmse"] = -cv_df["mean_test_score"]

        if round_col is not None:
            # Mean and standard deviation of the RMSE per halving round
            agg = cv_df.groupby(round_col)["rmse"].agg(["mean", "std", "count"]).reset_index()
            agg.columns = [round_col, "mean_rmse", "std_rmse", "n_cands"]

            ax.errorbar(agg[round_col], agg["mean_rmse"], yerr=agg["std_rmse"],
                        fmt="o-", color=PALETTE[0], capsize=4, linewidth=1.5,
                        markersize=6, label="Mean RMSE")

            # Annotate the number of candidates per round
            for _, row in agg.iterrows():
                ax.annotate(f"n={int(row['n_cands'])}",
                            (row[round_col], row["mean_rmse"]),
                            textcoords="offset points", xytext=(0, 12),
                            fontsize=8, ha="center", color=PALETTE[0])

            # Best RMSE line
            ax.axhline(res["best_score"], ls="--", color=PALETTE[1], lw=1,
                       label=f"Best RMSE = {res['best_score']:,.1f}")

            xlabel = "Halving round" if round_col == "iter" else "n_resources"
            ax.set_xlabel(xlabel)
        else:
            # Fallback: plot the sorted RMSE of all candidates
            sorted_rmse = cv_df["rmse"].sort_values().values
            ax.plot(range(len(sorted_rmse)), sorted_rmse, color=PALETTE[0], lw=1.5)
            ax.set_xlabel("Candidate rank")

        ax.set_ylabel("RMSE (kWh)")
        ax.set_title(f"Expert {expert_id}: {res['model_name']}")
        ax.legend(loc="upper right", framealpha=0.9)
        ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:,.0f}"))

    # Hide unused axes
    for idx in range(n_experts, nrows * ncols):
        axes[idx // ncols, idx % ncols].set_visible(False)

    fig.suptitle("HPO Convergence: Successive Halving per Expert",
                 fontsize=14, fontweight="bold", y=1.01)
    fig.tight_layout()
    fig.savefig(f"{RESULTS_DIR}/07_hpo_convergence_curves.png",
                bbox_inches="tight", facecolor="white")
    plt.show()
    print(f"Saved: {RESULTS_DIR}/07_hpo_convergence_curves.png")

### 4.2 Summary table: best hyperparameters per expert

The first table reports the best cross-validated RMSE and MAPE, the number of candidate evaluations and the run time per expert. The second table lists the selected hyperparameters (pipeline prefixes such as `reg__` are removed for readability).

In [ ]:
hpo_summary_df = pd.DataFrame(hpo_summary_rows)
display_cols = ["expert_id", "model", "best_rmse", "best_mape", "n_candidates_evaluated", "elapsed_s"]
print("HPO summary:")
display(hpo_summary_df[display_cols].style
        .format({"best_rmse": "{:,.2f}", "best_mape": "{:.2%}", "elapsed_s": "{:.1f}"})
        .set_caption("Best cross-validated RMSE and MAPE per expert after Halving Random Search")
        .set_table_styles([{"selector": "caption",
                            "props": [("font-weight", "bold"), ("font-size", "13px")]}]))

In [ ]:
# ── Detailed table of the best parameters ────────────────────────────────────
params_rows = []
for expert_id, res in sorted(hpo_results.items(), key=lambda x: int(x[0])):
    row = {"expert_id": expert_id, "model": res["model_name"]}
    for k, v in res["best_params"].items():
        # Clean the parameter name (remove the pipeline prefix)
        clean_k = k.split("__")[-1] if "__" in k else k
        row[clean_k] = round(v, 6) if isinstance(v, float) else v
    params_rows.append(row)

params_df = pd.DataFrame(params_rows)
print("\nDetailed hyperparameters per expert:")
display(params_df)

### 4.3 Hyperparameter distribution: parallel-coordinate plots

For each model family in use, a parallel-coordinate plot shows the 10 candidates with the lowest mean cross-validated RMSE of every expert that uses that family. Each numeric hyperparameter is min-max normalized to $[0, 1]$ across the plotted candidates. The plot shows which regions of the hyperparameter space are selected consistently across experts.

In [ ]:
# Parallel-coordinate plot for every model family present in the results
from pandas.plotting import parallel_coordinates

families_in_use = set(res["model_name"] for res in hpo_results.values())

for family in sorted(families_in_use):
    # Collect the top-10 candidates of every expert that uses this family
    top_candidates = []
    for expert_id, res in hpo_results.items():
        if res["model_name"] != family:
            continue
        cv_df = res["cv_results"].copy()
        cv_df["rmse"] = -cv_df["mean_test_score"]
        top10 = cv_df.nsmallest(10, "rmse")

        for _, row in top10.iterrows():
            p = row["params"]
            entry = {"expert": f"Expert {expert_id}"}
            for k, v in p.items():
                clean_k = k.split("__")[-1] if "__" in k else k
                if isinstance(v, (int, float)):
                    entry[clean_k] = v
            top_candidates.append(entry)

    if not top_candidates:
        continue

    pc_df = pd.DataFrame(top_candidates)
    numeric_cols = [c for c in pc_df.columns if c != "expert" and pc_df[c].dtype in ["float64", "int64"]]

    if len(numeric_cols) < 2:
        continue

    # Min-max normalize to [0, 1] for visualization
    pc_norm = pc_df.copy()
    for col in numeric_cols:
        cmin, cmax = pc_norm[col].min(), pc_norm[col].max()
        if cmax > cmin:
            pc_norm[col] = (pc_norm[col] - cmin) / (cmax - cmin)
        else:
            pc_norm[col] = 0.5

    fig, ax = plt.subplots(figsize=(max(8, len(numeric_cols) * 1.2), 4.5))
    parallel_coordinates(pc_norm[["expert"] + numeric_cols], "expert",
                         ax=ax, alpha=0.5, linewidth=1.5,
                         color=[PALETTE[i % len(PALETTE)] for i in range(pc_norm["expert"].nunique())])
    ax.set_title(f"Top-10 Hyperparameter Configurations: {family}", fontweight="bold")
    ax.set_ylabel("Normalized value")
    ax.legend(loc="upper right", fontsize=8, framealpha=0.9)
    plt.xticks(rotation=30, ha="right")
    fig.tight_layout()
    fig.savefig(f"{RESULTS_DIR}/07_hpo_parallel_{family}.png",
                bbox_inches="tight", facecolor="white")
    plt.show()
    print(f"Saved: {RESULTS_DIR}/07_hpo_parallel_{family}.png")

---

## 5. Gating HPO (threshold and temperature)

The adaptive gating evaluated here has two key parameters:

- **threshold** ($\theta$): confidence level above which the gating routes the input to a single expert (hard routing). Below this threshold, a soft (distance-weighted) combination of experts is used.
- **tau** ($\tau$): temperature of the softmax applied to the distances. A lower $\tau$ yields sharper (more concentrated) weight distributions; a higher $\tau$ yields a more uniform combination.

Since there are only two parameters with a small discrete grid ($7 \times 5 = 35$ combinations), an exhaustive grid search is appropriate; random search is not needed.

### Gating mechanism

The last 20% of the rows of the feature matrix (chronological split) form the validation set; the first 80% are used to fit a `StandardScaler` on the features and to compute one centroid per cluster (mean standardized feature vector of the cluster). For a validation input $\mathbf{x}$ (standardized with the same scaler):

1. Compute the Euclidean distance $d_k$ to the centroid of each cluster $k$.
2. Convert the distances into softmax weights: $w_k = \frac{\exp(-d_k / \tau)}{\sum_j \exp(-d_j / \tau)}$.
3. If $\max_k(w_k) \geq \theta$: **hard routing** to the expert $\arg\max_k(w_k)$.
4. Otherwise: **soft routing**, i.e. the weighted average $\hat{y} = \sum_k w_k \, \hat{y}_k / \sum_k w_k$ over the experts with $w_k \geq w_{\min}$ (`MIN_PROB` = 0.05).

The expert predictions $\hat{y}_k$ are produced by the best estimators found in Section 4, applied to the unscaled features (each linear expert has its own internal scaler).

The combination $(\theta, \tau)$ with the lowest MASE (Mean Absolute Scaled Error) on the validation set is selected. MASE scales the MAE by the MAE of a 24-hour seasonal naive forecast computed on the same validation series; since this denominator is identical for all grid points, ranking by MASE is equivalent to ranking by MAE. MAE, RMSE and MAPE are reported as well.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# 5.1  Prepare the validation set and the expert predictions
# ══════════════════════════════════════════════════════════════════════════
from sklearn.preprocessing import StandardScaler

# Use the last 20% of the rows as the validation set (chronological split)
val_frac   = 0.20
n_total    = len(features_df)
n_val      = int(n_total * val_frac)
val_start  = n_total - n_val

df_val = features_df.iloc[val_start:].copy()
print(f"Validation set: {len(df_val):,} samples ({val_frac:.0%} of the total)")
print(f"  Period: {df_val.index.min()} to {df_val.index.max()}")

# ── Standardize the features so that gating distances are meaningful ────────
scaler_gating = StandardScaler()
df_train = features_df.iloc[:val_start].copy()

# Fit the scaler on the training part and transform both training and validation parts
df_train_scaled = df_train.copy()
df_train_scaled[FEATURE_COLS] = scaler_gating.fit_transform(df_train[FEATURE_COLS])

df_val_scaled = df_val.copy()
df_val_scaled[FEATURE_COLS] = scaler_gating.transform(df_val[FEATURE_COLS])

# ── Cluster centroids (mean standardized feature vector per cluster) ────────
centroids = {}
for cid in sorted(features_df["cluster_id"].unique()):
    mask_c = df_train["cluster_id"] == cid
    centroids[cid] = df_train_scaled.loc[mask_c, FEATURE_COLS].mean().values

centroid_ids = sorted(centroids.keys())
centroid_matrix = np.vstack([centroids[c] for c in centroid_ids])  # (K, D)

# ── Expert predictions on the validation set ──────────────────
expert_preds_val = {}  # expert_id -> np.array of predictions

for expert_id, res in hpo_results.items():
    best_estimator = res["search_obj"].best_estimator_
    # The experts expect the original features (without the gating scaling),
    # since each linear expert has its own Pipeline with an internal StandardScaler.
    X_val_original = df_val[FEATURE_COLS].values
    expert_preds_val[expert_id] = best_estimator.predict(X_val_original)

y_val = df_val[TARGET].values
print(f"\nPredictions of {len(expert_preds_val)} experts generated on the validation set.")

In [ ]:
expert_preds_val

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# 5.2  Gating evaluation functions
# ══════════════════════════════════════════════════════════════════════════

def compute_gating_predictions(
    X_val: np.ndarray,
    expert_preds: dict,
    centroid_matrix: np.ndarray,
    centroid_ids: list,
    threshold: float,
    tau: float,
    min_prob: float = 0.05,
) -> np.ndarray:
    """
    Apply adaptive gating to combine the expert predictions.

    Parameters
    ----------
    X_val : validation feature matrix (N, D)
    expert_preds : dict expert_id -> predictions (N,)
    centroid_matrix : cluster centroids (K, D)
    centroid_ids : list of cluster ids matching the rows of centroid_matrix
    threshold : confidence threshold for hard routing
    tau : softmax temperature
    min_prob : minimum weight for an expert to take part in soft routing

    Returns
    -------
    y_pred : combined predictions (N,)
    """
    N = X_val.shape[0]
    y_pred = np.zeros(N)

    # Distances to the centroids: (N, K)
    from scipy.spatial.distance import cdist
    dists = cdist(X_val, centroid_matrix, metric="euclidean")

    for i in range(N):
        # Softmax weights with temperature
        d_i = dists[i]  # (K,)
        log_weights = -d_i / (tau + 1e-12)
        log_weights -= log_weights.max()  # numerical stability
        weights = np.exp(log_weights)
        weights /= weights.sum()

        if weights.max() >= threshold:
            # Hard routing
            best_k = centroid_ids[np.argmax(weights)]

            # Accept both integer and string keys in the dictionary
            if best_k in expert_preds:
                y_pred[i] = expert_preds[best_k][i]
            elif str(best_k) in expert_preds:
                y_pred[i] = expert_preds[str(best_k)][i]
            else:
                # Fallback: weighted average of the available experts
                total_w = 0.0
                for j, cid in enumerate(centroid_ids):
                    if cid in expert_preds:
                        y_pred[i] += weights[j] * expert_preds[cid][i]
                        total_w += weights[j]
                    elif str(cid) in expert_preds:
                        y_pred[i] += weights[j] * expert_preds[str(cid)][i]
                        total_w += weights[j]
                if total_w > 0:
                    y_pred[i] /= total_w
        else:
            # Soft routing: combine the experts with weight >= min_prob
            total_w = 0.0
            for j, cid in enumerate(centroid_ids):
                if weights[j] < min_prob:
                    continue
                if cid in expert_preds:
                    y_pred[i] += weights[j] * expert_preds[cid][i]
                    total_w += weights[j]
                elif str(cid) in expert_preds:
                    y_pred[i] += weights[j] * expert_preds[str(cid)][i]
                    total_w += weights[j]
            if total_w > 0:
                y_pred[i] /= total_w

    return y_pred


def mase(y_true: np.ndarray, y_pred: np.ndarray, seasonal_period: int = 24) -> float:
    """
    Mean Absolute Scaled Error.

    MAE scaled by the MAE of a seasonal naive forecast (period = 24 h)
    computed on the same series y_true.
    """
    mae_pred = np.mean(np.abs(y_true - y_pred))
    # Seasonal naive error
    naive_errors = np.abs(y_true[seasonal_period:] - y_true[:-seasonal_period])
    mae_naive = np.mean(naive_errors) if len(naive_errors) > 0 else 1.0
    return mae_pred / mae_naive if mae_naive > 0 else np.inf


print("Gating evaluation functions defined.")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# 5.3  Grid search over (threshold, tau)
# ══════════════════════════════════════════════════════════════════════════
from sklearn.metrics import mean_squared_error, mean_absolute_percentage_error

THRESHOLDS = [0.5, 0.6, 0.7, 0.75, 0.8, 0.85, 0.9]
TAUS       = [0.1, 0.5, 1.0, 2.0, 5.0]
MIN_PROB   = 0.05

# Use the STANDARDIZED features to compute the gating distances
X_val_feat = df_val_scaled[FEATURE_COLS].values

gating_grid_rows = []

print(f"Evaluating {len(THRESHOLDS) * len(TAUS)} (threshold, tau) combinations...")

for thr in THRESHOLDS:
    for tau_val in TAUS:
        y_pred_gated = compute_gating_predictions(
            X_val=X_val_feat,
            expert_preds=expert_preds_val,
            centroid_matrix=centroid_matrix,
            centroid_ids=centroid_ids,
            threshold=thr,
            tau=tau_val,
            min_prob=MIN_PROB,
        )
        mae_val  = mean_absolute_error(y_val, y_pred_gated)
        rmse_val = np.sqrt(mean_squared_error(y_val, y_pred_gated))
        mape_val = mean_absolute_percentage_error(y_val, y_pred_gated)
        mase_val = mase(y_val, y_pred_gated, seasonal_period=24)

        gating_grid_rows.append({
            "threshold": thr,
            "tau":       tau_val,
            "MAE":       mae_val,
            "RMSE":      rmse_val,
            "MAPE":      mape_val,
            "MASE":      mase_val,
        })

gating_grid_df = pd.DataFrame(gating_grid_rows)

# ── Best combination (lowest MASE) ─────────────────────────────────────────
best_gating_idx = gating_grid_df["MASE"].idxmin()
best_gating     = gating_grid_df.loc[best_gating_idx]

print(f"\nBest gating configuration:")
print(f"  threshold = {best_gating['threshold']}")
print(f"  tau       = {best_gating['tau']}")
print(f"  MAE       = {best_gating['MAE']:,.2f} kWh")
print(f"  RMSE      = {best_gating['RMSE']:,.2f} kWh")
print(f"  MAPE      = {best_gating['MAPE']:.4%}")
print(f"  MASE      = {best_gating['MASE']:.4f}")

In [ ]:
X_val_feat

In [ ]:
# ── 5.4  Heat maps: MASE and MAPE over the (threshold, tau) grid ────────────────
pivot_mase = gating_grid_df.pivot(index="threshold", columns="tau", values="MASE")
pivot_mape  = gating_grid_df.pivot(index="threshold", columns="tau", values="MAPE")

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# MASE heat map
sns.heatmap(pivot_mase, annot=True, fmt=".4f", cmap="YlOrRd",
            ax=axes[0], linewidths=0.5, linecolor="white",
            cbar_kws={"label": "MASE"})
axes[0].set_title("MASE: Gating Grid Search", fontweight="bold")
axes[0].set_xlabel(r"Temperature $\tau$")
axes[0].set_ylabel(r"Threshold $\theta$")

# Highlight the best cell
best_row = list(pivot_mase.index).index(best_gating["threshold"])
best_col = list(pivot_mase.columns).index(best_gating["tau"])
axes[0].add_patch(plt.Rectangle((best_col, best_row), 1, 1,
                                 fill=False, edgecolor="black", linewidth=3))

# MAPE heat map
sns.heatmap(pivot_mape, annot=True, fmt=",.4f", cmap="YlOrRd",
            ax=axes[1], linewidths=0.5, linecolor="white",
            cbar_kws={"label": "MAPE (fraction)"})
axes[1].set_title("MAPE: Gating Grid Search", fontweight="bold")
axes[1].set_xlabel(r"Temperature $\tau$")
axes[1].set_ylabel(r"Threshold $\theta$")

# Highlight the best cell
axes[1].add_patch(plt.Rectangle((best_col, best_row), 1, 1,
                                 fill=False, edgecolor="black", linewidth=3))

fig.suptitle("Gating Hyperparameter Optimization: Grid Search",
             fontsize=14, fontweight="bold", y=1.02)
fig.tight_layout()
fig.savefig(f"{RESULTS_DIR}/07_hpo_gating_heatmap.png",
            bbox_inches="tight", facecolor="white")
plt.show()
print(f"Saved: {RESULTS_DIR}/07_hpo_gating_heatmap.png")

In [ ]:
# ── 5.5  Full gating grid table ──────────────────────────────────────────
print("\nFull gating grid results (sorted by MASE):")
display(
    gating_grid_df
    .sort_values("MASE")
    .reset_index(drop=True)
    .style
    .format({"MAE": "{:,.2f}", "MASE": "{:.4f}", "MAPE": "{:.2%}", "RMSE": "{:,.2f}"})
    .highlight_min(subset=["MASE"], color="#d4edda")
    .set_caption("Gating grid search results")
)

---

## 6. Persist the optimized artifacts

The following files are saved:

1. **Best hyperparameters** per expert, as JSON (`models/hyperparameters/best_hyperparameters.json`). Note: the field `best_MAE` of this file stores the best cross-validated **RMSE** of Section 4 (the field name is kept for compatibility).
2. **Optimized models** (the refitted `best_estimator_` of each search) as pickle files (`models/hyperparameters/expert_<id>_optimised.pkl`).
3. **HPO results table** as CSV (`results/07_hpo_results.csv`).
4. **Optimal gating parameters** as JSON (`models/hyperparameters/best_gating_params.json`) and the full gating grid as CSV (`results/07_gating_grid_results.csv`).

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# 6.1  Save the best hyperparameters (JSON)
# ══════════════════════════════════════════════════════════════════════════
hp_dir = Path(MODELS_DIR) / "hyperparameters"

best_params_dict = {}
for expert_id, res in hpo_results.items():
    # Convert NumPy types for JSON serialization
    params_clean = {}
    for k, v in res["best_params"].items():
        if isinstance(v, (np.integer,)):
            params_clean[k] = int(v)
        elif isinstance(v, (np.floating,)):
            params_clean[k] = float(v)
        else:
            params_clean[k] = v
    best_params_dict[expert_id] = {
        "model": res["model_name"],
        "best_MAE": float(res["best_score"]),  # note: stores the best CV RMSE (key name kept for compatibility)
        "params": params_clean,
    }

hp_path = hp_dir / "best_hyperparameters.json"
with open(hp_path, "w") as f:
    json.dump(best_params_dict, f, indent=2)
print(f"Best hyperparameters saved: {hp_path}")

# ══════════════════════════════════════════════════════════════════════════
# 6.2  Save the optimized models (pickle)
# ══════════════════════════════════════════════════════════════════════════

for expert_id, res in hpo_results.items():
    model_path = hp_dir / f"expert_{expert_id}_optimised.pkl"
    with open(model_path, "wb") as f:
        pickle.dump(res["search_obj"].best_estimator_, f)
    print(f"  Saved: {model_path}")

# ══════════════════════════════════════════════════════════════════════════
# 6.3  Save the HPO results table (CSV)
# ══════════════════════════════════════════════════════════════════════════

hpo_csv_path = Path(RESULTS_DIR) / "07_hpo_results.csv"
hpo_summary_df.to_csv(hpo_csv_path, index=False)
print(f"\nHPO summary saved: {hpo_csv_path}")

# ══════════════════════════════════════════════════════════════════════════
# 6.4  Save the optimal gating parameters (JSON)
# ══════════════════════════════════════════════════════════════════════════

gating_best = {
    "threshold": float(best_gating["threshold"]),
    "tau":       float(best_gating["tau"]),
    "min_prob":  MIN_PROB,
    "MAE":       float(best_gating["MAE"]),
    "MASE":      float(best_gating["MASE"]),
    "MAPE":      float(best_gating["MAPE"]),
    "RMSE":      float(best_gating["RMSE"])
}

gating_path = hp_dir / "best_gating_params.json"
with open(gating_path, "w") as f:
    json.dump(gating_best, f, indent=2)
print(f"Best gating parameters saved: {gating_path}")

# ── Save the full gating grid ─────────────────────────────────────────────────
gating_csv_path = Path(RESULTS_DIR) / "07_gating_grid_results.csv"
gating_grid_df.to_csv(gating_csv_path, index=False)
print(f"Gating grid saved: {gating_csv_path}")